# Capa 2: Depuración, Deduplicación y Reglas DULCINEA (Silver)
**Objetivo:** Cargar el CSV Bronze, realizar deduplicación por paciente, imputación de nulos, estandarización de fechas y aplicar las reglas del modelo de abandono DULCINEA.

Carga de Datos y Creación de Nombre Completo

In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_BRONZE = BASE_DIR / "data" / "bronze"
DIR_SILVER = BASE_DIR / "data" / "silver"
DIR_SILVER.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DIR_BRONZE / "cohorte_vih_bronze.csv")

# Renombrar columna de fecha técnica si aplica
for col in df.columns:
    if "2025-08-08" in str(col):
        df.rename(columns={col: "Fecha_Corte_Reporte_CAC"}, inplace=True)

# Imputar Segundo Nombre con espacio en blanco para construir Nombre Completo
df["Primer Nombre"] = df["Primer Nombre"].fillna("")
df["Segundo Nombre"] = df["Segundo Nombre"].fillna("")
df["Primer Apellido"] = df["Primer Apellido"].fillna("")
df["Segundo Apellido"] = df["Segundo Apellido"].fillna("")

df["Nombre_Completo"] = (
    df["Primer Nombre"]
    + " "
    + df["Segundo Nombre"]
    + " "
    + df["Primer Apellido"]
    + " "
    + df["Segundo Apellido"]
)
df["Nombre_Completo"] = (
    df["Nombre_Completo"].str.replace(r"\s+", " ", regex=True).str.strip()
)

# Validación Celda 2
print(f"Registros iniciales Silver: {len(df)}")
print("Muestra de Nombres Completos:")
print(df[["Numero de Identificacion", "Nombre_Completo"]].head(3))

Registros iniciales Silver: 850
Muestra de Nombres Completos:
   Numero de Identificacion              Nombre_Completo
0                 108847493  Carlos Andrés Ramírez Pérez
1                  87490893         Miguel Romero Castro
2                  38538251             Ana Muñoz Vargas


Deduplicación de Pacientes

In [2]:
# Crear Llave Única por Paciente
df["Llave_Paciente"] = (
    df["TD"].astype(str)
    + "_"
    + df["Numero de Identificacion"].astype(str)
    + "_"
    + df["Nombre_Completo"]
)

# Detectar duplicados
duplicados = df[df.duplicated(subset=["Llave_Paciente"], keep=False)]
num_duplicados = len(duplicados)

# Convertir fecha diagnóstico para conservar el reporte más reciente si hubiese duplicados
df["36. Fecha del diagnostico de VIH"] = pd.to_datetime(
    df["36. Fecha del diagnostico de VIH"], errors="coerce"
)
df = df.sort_values(
    by=["Llave_Paciente", "36. Fecha del diagnostico de VIH"],
    ascending=[True, False],
)

# Eliminar duplicados reteniendo el primero (el más reciente)
df_silver = df.drop_duplicates(subset=["Llave_Paciente"], keep="first").copy()

# Validación Celda 3
print(f"Registros duplicados encontrados: {num_duplicados}")
print(f"Registros únicos tras la deduplicación: {len(df_silver)}")

Registros duplicados encontrados: 0
Registros únicos tras la deduplicación: 850


Imputación de Faltantes e Inconsistencias

In [3]:
# Variables Categóricas e Inmunológicas
df_silver["Poblacion Clave"] = df_silver["Poblacion Clave"].fillna(
    "SIN INFORMACION"
)
df_silver["11. Pertenencia étnica"] = df_silver["11. Pertenencia étnica"].fillna(
    "SIN INFORMACION"
)

# Regla Condicional de Gestación:
# 1. Asignar 0 a Hombres y No Gestantes
condicion_no_gestante = (df_silver["Sexo"] == "M") | (
    df_silver["GESTANTE"].astype(str).str.upper() == "NO GESTANTE"
)
df_silver.loc[condicion_no_gestante, "Edad gestacional"] = 0

# 2. Imputar 0 a los casos gestantes sin registro de semanas de gestación
df_silver["Edad gestacional"] = df_silver["Edad gestacional"].fillna(0)

# Validación Celda 4
print("Verificación de nulos en variables clave (debe dar 0 en todas):")
print(
    df_silver[
        ["Poblacion Clave", "11. Pertenencia étnica", "Edad gestacional"]
    ].isnull().sum()
)

Verificación de nulos en variables clave (debe dar 0 en todas):
Poblacion Clave           0
11. Pertenencia étnica    0
Edad gestacional          0
dtype: int64


Estandarización de Fechas y Tipos

In [4]:
# Convertir Numero de Identificacion a Texto
df_silver["Numero de Identificacion"] = df_silver[
    "Numero de Identificacion"
].astype(str)

# Convertir todas las columnas de fechas a formato datetime (YYYY-MM-DD)
columnas_fechas = [
    c
    for c in df_silver.columns
    if "fecha" in c.lower() or "fec" in c.lower() or "nacimiento" in c.lower()
]
for col in columnas_fechas:
    df_silver[col] = pd.to_datetime(df_silver[col], errors="coerce")

# Validación Celda 5
print(f"Total de columnas con formato Fecha estandarizadas: {len(columnas_fechas)}")

Total de columnas con formato Fecha estandarizadas: 37


Clasificación del Abandono (Regla DULCINEA)

In [5]:
# Categorización de Carga Viral
def categorizar_cv(val):
    val_str = str(val).strip().lower()
    if "indetectable" in val_str or val_str in ["<50", "0"]:
        return "Indetectable (<50 copias/ml)"
    try:
        num = float(val)
        return (
            "Indetectable (<50 copias/ml)"
            if num < 50
            else "Detectable (>=50 copias/ml)"
        )
    except ValueError:
        return "Sin Dato / En Proceso"


df_silver["Categoria_Carga_Viral"] = df_silver[
    "76.1 Resultado de la última Carga viral para VIH"
].apply(categorizar_cv)


# Clasificación del Abandono y Estado del Tratamiento
def clasificar_estado_dulcinea(row):
    recibe_tar = str(row.get("77. Recibe TAR", "")).strip().capitalize()
    cv = row["Categoria_Carga_Viral"]

    if recibe_tar == "No":
        return "Abandono / Sin TAR"
    elif recibe_tar == "Sí":
        if "Indetectable" in cv:
            return "En TAR - Virológicamente Controlado"
        else:
            return "En TAR - Con Carga Viral Detectable"
    return "Seguimiento Pendiente"


df_silver["Estado_Seguimiento_VIH"] = df_silver.apply(
    clasificar_estado_dulcinea, axis=1
)
df_silver["es_abandono"] = (
    df_silver["Estado_Seguimiento_VIH"] == "Abandono / Sin TAR"
).astype(int)

# Validación Celda 6
print("Distribución del Estado de Seguimiento (Regla DULCINEA):")
print(df_silver["Estado_Seguimiento_VIH"].value_counts())

Distribución del Estado de Seguimiento (Regla DULCINEA):
Estado_Seguimiento_VIH
En TAR - Virológicamente Controlado    451
Abandono / Sin TAR                     217
En TAR - Con Carga Viral Detectable    182
Name: count, dtype: int64


Ordenamiento y Exportación Final Silver

In [6]:
# Ordenar por Subregión, Municipio y Fecha Diagnóstico Descendente
df_silver = df_silver.sort_values(
    by=["Subregion", "Municipio residencia", "36. Fecha del diagnostico de VIH"],
    ascending=[True, True, False],
)

# Exportar a CSV y Parquet en data/silver/
ruta_csv_silver = DIR_SILVER / "cohorte_vih_silver.csv"
ruta_parquet_silver = DIR_SILVER / "cohorte_vih_silver.parquet"

df_silver.to_csv(ruta_csv_silver, index=False, encoding="utf-8-sig")
df_silver.to_parquet(ruta_parquet_silver, index=False)

# Validación Celda 7
print("CAPA SILVER FINALIZADA CON ÉXITO")
print(f"Registros guardados: {len(df_silver):,}")
print(f"Ruta CSV:     {ruta_csv_silver}")
print(f"Ruta Parquet: {ruta_parquet_silver}")

CAPA SILVER FINALIZADA CON ÉXITO
Registros guardados: 850
Ruta CSV:     c:\Users\DRODRIGUEZQU\Downloads\proyecto_dulcinea_vih_mejorado\proyecto_dulcinea_vih\data\silver\cohorte_vih_silver.csv
Ruta Parquet: c:\Users\DRODRIGUEZQU\Downloads\proyecto_dulcinea_vih_mejorado\proyecto_dulcinea_vih\data\silver\cohorte_vih_silver.parquet
